# 09주차 실습: CTR 모델과 Pareto 의사결정

강의: [PDF](../pdf/week09_multi_objective_optimization.pdf) · **PDF 실제 페이지 6–23** (인쇄 번호와 다를 수 있음)

## 학습 목표
성능·처리 시간·정책 비용의 목적 방향을 통일하고 Pareto 후보를 찾는다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

## 기존 실습 연결
기본 실습 후 [week09_ctr_multi_objective_optimization_notebook.ipynb](week09_ctr_multi_objective_optimization_notebook.ipynb)로 이어갑니다. 호환 방법과 수정 사항은 COMPATIBILITY.md를 확인하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 모델 후보 비교

In [ ]:
import time
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from labkit.common import ctr_pipeline, split_ctr, pareto_mask
df=pd.read_csv(DATA/'ctr.csv'); train,valid,test=split_ctr(df)
rows=[]; models=[]
for c in [.01,.1,1,10]:
    model=ctr_pipeline().set_params(model__C=c).fit(train,train.clicked)
    model.predict_proba(valid)  # warm-up
    times=[]
    for _ in range(5):
        t=time.perf_counter(); pred=model.predict_proba(valid)[:,1]; times.append(time.perf_counter()-t)
    rows.append({'C':c,'auc':roc_auc_score(valid.clicked,pred),'inference_ms':np.median(times)*1000})
    models.append(model)
results=pd.DataFrame(rows)
results['pareto']=pareto_mask(results[['auc','inference_ms']].to_numpy()*[-1,1])
display(results)
assert pareto_mask([[1,2],[1,2],[2,3]]).tolist()==[True,True,False]

## 임계값과 정책 비용

In [ ]:
best=models[int(results.auc.argmax())]; probabilities=best.predict_proba(valid)[:,1]
policy=[]
for threshold in np.linspace(.1,.9,9):
    chosen=probabilities>=threshold
    policy.append({'threshold':threshold,'precision':precision_score(valid.clicked,chosen,zero_division=0),
      'recall':recall_score(valid.clicked,chosen), 'total_bid':valid.loc[chosen,'bid_price'].sum(), 'selected':int(chosen.sum())})
policies=pd.DataFrame(policy)
policies['pareto']=pareto_mask(policies[['precision','recall','total_bid']].to_numpy()*[-1,-1,1])
display(policies)
ax=results.plot.scatter(x='inference_ms',y='auc',c=results.pareto.astype(int),colormap='viridis'); plt.show()
out=output_dir('week09'); results.to_csv(out/'model_pareto.csv',index=False); policies.to_csv(out/'threshold_pareto.csv',index=False)
print('test는 최종 정책 확정 후 한 번만 사용합니다.')

## 확장 과제 및 제출
기존 노트북의 NSGA-II 단계로 이어가세요. 비지배 정렬·crowding·부모/자식 통합 선택을 설명하고 세 seed로 비교하세요. total_bid는 단순 정책 비용 proxy이며 실제 집행 비용이 아닙니다. popularity 표준편차를 공정성 전체로 해석하지 마세요.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.